# RMFS Re-Clustering Pipeline

**Phase 0** Initial k-means clustering on static features  
**Phase 1** Simulate first half of duration  
**Mid-point** Extract simulation features → re-cluster SKUs  
**Phase 2** Simulate second half with updated pod assignments  
**Results** Compare Phase 1 vs Phase 2 metrics

## Configuration

In [ ]:
TOTAL_HOURS = 0.25   # Total simulation duration in hours
K = 5                # Number of k-means clusters

# Item cluster order frequency configuration (must sum to 1.0)
ITEMS_ORDERS_CLASS_CONFIG = {
    4: 0.45,  # 45%
    0: 0.25,  # 25%
    2: 0.20,  # 20%
    1: 0.10,  # 10%
    3: 0.05,  #  5%
}

## Setup

In [5]:
import os, sys, io, contextlib, time
import pandas as pd
import numpy as np
from math import ceil, sqrt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import tqdm

# Resolve project paths
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
# Walk up until we find the project root (contains sku_sample.csv)
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
PIPELINE_DIR = os.path.join(PROJECT_ROOT, "pipeline")
NETLOGO_DIR = os.path.join(PROJECT_ROOT, "netlogo")

sys.path.insert(0, PIPELINE_DIR)
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, NETLOGO_DIR)

from mid_sim_features import extract_all_sim_features

# Constants
LEAD_TIME = 1.0
SERVICE_LEVEL_Z = 1.2816  # 90% service level
RANDOM_STATE = 42
N_INIT = 20
CLUSTER_FEATURES = ["mean_demand", "cv_demand", "demand_frequency", "avg_affinity", "max_affinity"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Total hours:  {TOTAL_HOURS}h, K={K}")
print(f"Phase 1: {TOTAL_HOURS/2}h → re-cluster → Phase 2: {TOTAL_HOURS/2}h")

Project root: /Users/brendantm/Taiwan/TEEP/salsa-rmfs
Total hours:  0.25h, K=5
Phase 1: 0.125h → re-cluster → Phase 2: 0.125h


In [6]:
@contextlib.contextmanager
def suppress_stdout():
    """Redirect stdout to devnull to silence simulation debug prints."""
    old = sys.stdout
    sys.stdout = io.StringIO()
    try:
        yield
    finally:
        sys.stdout = old


def run_clustering(sku_df, k):
    X = sku_df[CLUSTER_FEATURES].copy()
    X["mean_demand"] = np.log1p(X["mean_demand"])
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
    sku_df["cluster"] = km.fit_predict(X_scaled)
    return sku_df


def compute_initial_inventory(mean_demand, std_demand):
    mu = 0 if pd.isna(mean_demand) else float(mean_demand)
    sigma = 0 if pd.isna(std_demand) else float(std_demand)
    return int(ceil(mu + SERVICE_LEVEL_Z * sigma * sqrt(LEAD_TIME)))


def run_phase(target_tick, phase_name):
    """Run tick() loop with a tqdm progress bar."""
    from netlogo import tick as sim_tick

    tick_count = 0
    current_tick = 0.0
    metrics_log = []
    t0 = time.time()

    pbar = tqdm(total=int(target_tick), desc=phase_name, unit="s", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}s [{elapsed}<{remaining}, {postfix}]")

    while current_tick < target_tick:
        with suppress_stdout():
            result = sim_tick()

        if isinstance(result, str):
            print(f"ERROR: {result}")
            break

        prev_tick = current_tick
        current_tick = result[-1]
        tick_count += 1

        metrics_log.append({
            "tick": current_tick,
            "total_energy": result[1],
            "job_queue_len": result[2],
            "stop_and_go": result[3],
            "total_turning": result[4],
            "orders_finished": result[6],
        })

        pbar.update(int(current_tick) - pbar.n)
        if tick_count % 50 == 0:
            pbar.set_postfix(orders=result[6])

    pbar.update(int(target_tick) - pbar.n)
    pbar.set_postfix(orders=metrics_log[-1]["orders_finished"] if metrics_log else 0)
    pbar.close()

    elapsed = time.time() - t0
    metrics_df = pd.DataFrame(metrics_log)
    print(f"  {phase_name} complete: {tick_count} ticks in {elapsed:.1f}s")

    return {"final_tick": current_tick, "tick_count": tick_count, "metrics": metrics_df}


def compute_extra_metrics(orders_finished, generated_order_path, pod_info_path):
    """Compute order throughput, replenishment/pick ratio, and pod utilization."""
    metrics = {}

    orders_generated = 0
    if os.path.exists(generated_order_path):
        gen_df = pd.read_csv(generated_order_path, dtype=str)
        orders_generated = gen_df["order_id"].nunique()
    metrics["orders_generated"] = orders_generated
    metrics["order_throughput"] = orders_finished / orders_generated if orders_generated > 0 else 0.0

    total_picks = 0
    total_replenishments = 0
    total_units_picked = 0
    pod_visits = 0
    if os.path.exists(pod_info_path):
        pod_df = pd.read_csv(pod_info_path)
        if not pod_df.empty and "task_type" in pod_df.columns:
            pick_df = pod_df[pod_df["task_type"] == 1]
            replen_df = pod_df[pod_df["task_type"] == 2]
            total_picks = len(pick_df)
            total_replenishments = len(replen_df)
            total_units_picked = pick_df["qty"].sum() if "qty" in pick_df.columns else 0
            if not pick_df.empty and "pod_id" in pick_df.columns and "processed_time" in pick_df.columns:
                pod_visits = pick_df.groupby(["pod_id", "processed_time"]).ngroups

    metrics["total_picks"] = total_picks
    metrics["total_replenishments"] = total_replenishments
    metrics["replenishment_pick_ratio"] = total_replenishments / total_picks if total_picks > 0 else 0.0
    metrics["total_units_picked"] = total_units_picked
    metrics["pod_visits"] = pod_visits
    metrics["pod_utilization"] = total_units_picked / pod_visits if pod_visits > 0 else 0.0

    return metrics

print("Helper functions defined.")

Helper functions defined.


## Phase 0: Initial K-Means Clustering

In [7]:
sku_sample_path = os.path.join(PROJECT_ROOT, "sku_sample.csv")

sku_df = pd.read_csv(sku_sample_path)
sku_df["item_code"] = sku_df["item_code"].astype(str)
for col in CLUSTER_FEATURES:
    sku_df[col] = pd.to_numeric(sku_df[col], errors="coerce").fillna(0)

sku_df = run_clustering(sku_df, K)
sku_df.to_csv(sku_sample_path, index=False)

dist = sku_df["cluster"].value_counts().sort_index()
print(f"Clustered {len(sku_df)} SKUs into {K} groups:")
for cluster_id, count in dist.items():
    print(f"  Cluster {cluster_id}: {count} SKUs")

Clustered 1448 SKUs into 5 groups:
  Cluster 0: 414 SKUs
  Cluster 1: 241 SKUs
  Cluster 2: 259 SKUs
  Cluster 3: 59 SKUs
  Cluster 4: 475 SKUs


## Initialize Simulation & Generate Orders

In [ ]:
total_seconds = TOTAL_HOURS * 3600
half_seconds = total_seconds / 2

os.chdir(NETLOGO_DIR)
sku_sample_rel = os.path.relpath(sku_sample_path, NETLOGO_DIR)

# Generate items.csv and pods.csv via thesis-based FFD assignment
from assignment.convert_to_sim import run_full_conversion
run_full_conversion(
    base_dir=PROJECT_ROOT,
    netlogo_dir=NETLOGO_DIR,
    sku_sample_path=sku_sample_path,
)

from netlogo import reload_data_for_phase, reload_pods_only

total_order_hours = max(1, int(np.ceil(TOTAL_HOURS)))
backlog_order_hours = total_order_hours + 1

print(f"Generating orders for {total_order_hours}h and initializing simulation...")
with suppress_stdout():
    reload_data_for_phase(
        sku_sample_path=sku_sample_rel,
        order_period_hours=total_order_hours,
        backlog_period_hours=backlog_order_hours,
        items_orders_class_configuration=ITEMS_ORDERS_CLASS_CONFIG
    )
print("Simulation initialized.")

## Phase 1: First Half

In [9]:
phase1 = run_phase(half_seconds, "Phase 1")

# Save Phase 1 orders
results_dir = os.path.join(PROJECT_ROOT, "results")
os.makedirs(results_dir, exist_ok=True)

phase1_order_finished = None
if os.path.exists("order-finished.csv"):
    phase1_order_finished = pd.read_csv("order-finished.csv")
    phase1_order_finished.to_csv(os.path.join(results_dir, "phase1_orders.csv"), index=False)

# Phase 1 extra metrics
phase1_finished_count = int(phase1["metrics"]["orders_finished"].iloc[-1]) if not phase1["metrics"].empty else 0
phase1_extra = compute_extra_metrics(phase1_finished_count, "generated_order.csv", "pod_info.csv")

Phase 1:   0%|          | 0/450s [00:00<?, ]

  Phase 1 complete: 3001 ticks in 60.0s


## Drain: Finish In-Flight Work Before Re-Clustering

Stop accepting new orders and let all robots finish their current jobs.
This ensures no work is lost during the state reset.

In [ ]:
from netlogo import drain_simulation

print("Draining simulation before re-clustering...")
drain_ticks, drain_final_tick, drain_orders = drain_simulation()

print(f"\nDrain result: {drain_ticks} extra ticks, "
      f"final_tick={drain_final_tick:.1f}s, "
      f"total orders finished={drain_orders}")

# Use the actual midpoint time (after drain) for order trimming
midpoint_seconds = drain_final_tick

## Mid-Point: Extract Features & Re-Cluster

In [10]:
# Extract simulation features
print("Extracting simulation features...")
sim_features = extract_all_sim_features(
    pod_info_path="pod_info.csv",
    assign_order_path="assign_order.csv",
    time_window_seconds=3600
)
print(f"Extracted features for {len(sim_features)} SKUs")

# Re-cluster
sku_df = pd.read_csv(sku_sample_path)
sku_df["item_code"] = sku_df["item_code"].astype(str)
sku_df["cluster_phase1"] = sku_df["cluster"]

if not sim_features.empty:
    sim_features["item_code"] = sim_features["item_code"].astype(str)
    observed_skus = set(sim_features["item_code"].values)
    print(f"Observed {len(observed_skus)} / {len(sku_df)} SKUs in simulation")

    sku_df = sku_df.merge(sim_features, on="item_code", how="left")

    for static_col, sim_col in [
        ("mean_demand", "sim_mean_demand"),
        ("cv_demand", "sim_cv_demand"),
        ("demand_frequency", "sim_frequency"),
        ("avg_affinity", "sim_avg_affinity"),
        ("max_affinity", "sim_max_affinity"),
    ]:
        if sim_col in sku_df.columns:
            mask = sku_df["item_code"].isin(observed_skus) & sku_df[sim_col].notna()
            sku_df.loc[mask, static_col] = sku_df.loc[mask, sim_col]

    if "sim_std_demand" in sku_df.columns:
        mask = sku_df["item_code"].isin(observed_skus) & sku_df["sim_std_demand"].notna()
        sku_df.loc[mask, "std_demand"] = sku_df.loc[mask, "sim_std_demand"]

    sim_cols = [c for c in sku_df.columns if c.startswith("sim_")]
    sku_df = sku_df.drop(columns=sim_cols)

for col in CLUSTER_FEATURES:
    sku_df[col] = pd.to_numeric(sku_df[col], errors="coerce").fillna(0)

sku_df = run_clustering(sku_df, K)

sku_df["item_initial_inventory"] = sku_df.apply(
    lambda row: compute_initial_inventory(row["mean_demand"], row["std_demand"]),
    axis=1
)
if "number_of_item_in_a_box" in sku_df.columns:
    sku_df["box_initial_inventory"] = sku_df.apply(
        lambda row: int(ceil(row["item_initial_inventory"] / max(row["number_of_item_in_a_box"], 1)))
        if pd.notna(row["number_of_item_in_a_box"]) and row["number_of_item_in_a_box"] > 0
        else 0,
        axis=1
    )

changed = (sku_df["cluster"] != sku_df["cluster_phase1"]).sum()
sku_df = sku_df.drop(columns=["cluster_phase1"])
sku_df.to_csv(sku_sample_path, index=False)

dist = sku_df["cluster"].value_counts().sort_index()
print(f"Re-clustered: {changed} SKUs changed cluster")
for cluster_id, count in dist.items():
    print(f"  Cluster {cluster_id}: {count} SKUs")

Extracting simulation features...
Extracted features for 77 SKUs
Observed 77 / 1448 SKUs in simulation
Re-clustered: 0 SKUs changed cluster
  Cluster 0: 414 SKUs
  Cluster 1: 241 SKUs
  Cluster 2: 259 SKUs
  Cluster 3: 59 SKUs
  Cluster 4: 475 SKUs


## Phase 2: Second Half (After Re-Clustering)

In [ ]:
# Regenerate pods.csv with new cluster assignments (items.csv stable due to sorted item_code)
run_full_conversion(
    base_dir=PROJECT_ROOT,
    netlogo_dir=NETLOGO_DIR,
    sku_sample_path=sku_sample_path,
)

# Reload simulation with new pod assignments
print("Reloading simulation with new clusters...")
with suppress_stdout():
    reload_pods_only(
        sku_sample_path=sku_sample_rel,
        midpoint_seconds=midpoint_seconds
    )
print("Simulation reloaded with new pod assignments.")

# Run Phase 2
phase2 = run_phase(half_seconds, "Phase 2")

# Save Phase 2 orders
phase2_order_finished = None
if os.path.exists("order-finished.csv"):
    phase2_order_finished = pd.read_csv("order-finished.csv")
    phase2_order_finished.to_csv(os.path.join(results_dir, "phase2_orders.csv"), index=False)

# Phase 2 extra metrics
phase2_finished_count = int(phase2["metrics"]["orders_finished"].iloc[-1]) if not phase2["metrics"].empty else 0
phase2_extra = compute_extra_metrics(phase2_finished_count, "generated_order.csv", "pod_info.csv")

## Results

In [12]:
# Save tick metrics
if not phase1["metrics"].empty:
    phase1["metrics"]["phase"] = 1
if not phase2["metrics"].empty:
    phase2["metrics"]["phase"] = 2
all_metrics = pd.concat([phase1["metrics"], phase2["metrics"]], ignore_index=True)
all_metrics.to_csv(os.path.join(results_dir, "tick_metrics.csv"), index=False)

# Build summary
def _last(metrics_df, col):
    return metrics_df[col].iloc[-1] if not metrics_df.empty else 0

summary = {
    "total_hours": TOTAL_HOURS, "k_clusters": K,
    "phase1_ticks": phase1["tick_count"], "phase2_ticks": phase2["tick_count"],
    "phase1_orders_finished": _last(phase1["metrics"], "orders_finished"),
    "phase2_orders_finished": _last(phase2["metrics"], "orders_finished"),
    "phase1_final_energy": _last(phase1["metrics"], "total_energy"),
    "phase2_final_energy": _last(phase2["metrics"], "total_energy"),
    "phase1_stop_and_go": _last(phase1["metrics"], "stop_and_go"),
    "phase2_stop_and_go": _last(phase2["metrics"], "stop_and_go"),
    "phase1_total_turning": _last(phase1["metrics"], "total_turning"),
    "phase2_total_turning": _last(phase2["metrics"], "total_turning"),
    "phase1_peak_job_queue": phase1["metrics"]["job_queue_len"].max() if not phase1["metrics"].empty else 0,
    "phase2_peak_job_queue": phase2["metrics"]["job_queue_len"].max() if not phase2["metrics"].empty else 0,
}
for prefix, extra in [("phase1", phase1_extra), ("phase2", phase2_extra)]:
    summary[f"{prefix}_orders_generated"] = extra["orders_generated"]
    summary[f"{prefix}_order_throughput"] = round(extra["order_throughput"], 4)
    summary[f"{prefix}_total_picks"] = extra["total_picks"]
    summary[f"{prefix}_total_replenishments"] = extra["total_replenishments"]
    summary[f"{prefix}_replenishment_pick_ratio"] = round(extra["replenishment_pick_ratio"], 4)
    summary[f"{prefix}_total_units_picked"] = extra["total_units_picked"]
    summary[f"{prefix}_pod_visits"] = extra["pod_visits"]
    summary[f"{prefix}_pod_utilization"] = round(extra["pod_utilization"], 4)

pd.DataFrame([summary]).to_csv(os.path.join(results_dir, "summary.csv"), index=False)

# Display comparison
print("=" * 60)
print(f"  RESULTS — {TOTAL_HOURS}h simulation, K={K}")
print("=" * 60)
for label, phase_data, orders_df, extra in [
    ("Phase 1 (before re-cluster)", phase1, phase1_order_finished, phase1_extra),
    ("Phase 2 (after re-cluster)", phase2, phase2_order_finished, phase2_extra),
]:
    m = phase_data["metrics"]
    print(f"\n  {label}")
    if not m.empty:
        print(f"    Orders finished:    {m['orders_finished'].iloc[-1]}")
        print(f"    Total energy:       {m['total_energy'].iloc[-1]:.2f}")
        print(f"    Stop & go:          {m['stop_and_go'].iloc[-1]}")
        print(f"    Total turning:      {m['total_turning'].iloc[-1]}")
        print(f"    Peak job queue:     {m['job_queue_len'].max()}")
        print(f"    Avg job queue:      {m['job_queue_len'].mean():.1f}")
    if orders_df is not None and not orders_df.empty:
        if "order_complete_time" in orders_df.columns and "process_start_time" in orders_df.columns:
            ct = orders_df["order_complete_time"] - orders_df["process_start_time"]
            print(f"    Avg cycle time:     {ct.mean():.1f}s")
            print(f"    Max cycle time:     {ct.max():.1f}s")
    print(f"    Order throughput:   {extra['order_throughput']:.4f} ({extra['orders_generated']} generated)")
    print(f"    Replen/pick ratio:  {extra['replenishment_pick_ratio']:.4f} ({extra['total_replenishments']}R / {extra['total_picks']}P)")
    print(f"    Pod utilization:    {extra['pod_utilization']:.4f} ({extra['total_units_picked']} units / {extra['pod_visits']} visits)")

print(f"\nResults saved to {results_dir}/")

  RESULTS — 0.25h simulation, K=5

  Phase 1 (before re-cluster)
    Orders finished:    12
    Total energy:       812994.91
    Stop & go:          1130
    Total turning:      778
    Peak job queue:     46
    Avg job queue:      23.5
    Order throughput:   0.0755 (159 generated)
    Replen/pick ratio:  0.0000 (0R / 74P)
    Pod utilization:    12.0244 (493 units / 41 visits)

  Phase 2 (after re-cluster)
    Orders finished:    12
    Total energy:       799315.77
    Stop & go:          1008
    Total turning:      754
    Peak job queue:     25
    Avg job queue:      11.1
    Order throughput:   0.1154 (104 generated)
    Replen/pick ratio:  0.0789 (3R / 38P)
    Pod utilization:    6.6842 (254 units / 38 visits)

Results saved to /Users/brendantm/Taiwan/TEEP/salsa-rmfs/results/
